# k-Nearest Neighbours on Iris

**Goal:** run the full $k$-NN workflow on the smallest standard classification dataset.
Iris has 150 flowers, four measurements, and three species, so every step finishes in a
fraction of a second and the decision boundary can be drawn exactly.

The workflow is the same one used on MNIST and ImageNet in the companion notebooks:

1. load and inspect the data,
2. split into train and test **before** any preprocessing,
3. standardise the features,
4. choose $k$ by cross-validation,
5. evaluate on the held-out test set and inspect the confusion matrix.

**What to expect:** setosa is linearly separable from the other two species in the petal
features; versicolor and virginica overlap slightly. A good model reaches the low-to-mid
90s. With only 45 test flowers, a single error moves accuracy by more than two percentage
points, so treat small differences with care.

## Setup

Only NumPy, matplotlib, and scikit-learn are required. The setup is split into two
cells: the first imports the lightweight libraries and fixes the random seed, the second
imports matplotlib and scikit-learn. On a fresh machine the second cell can take
10--30 seconds because matplotlib builds its font cache once.

In [ ]:
import numpy as np
from time import perf_counter

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Core setup complete.")

The next cell imports matplotlib and scikit-learn. The progress messages show which
import is running.

In [ ]:
print("Importing matplotlib (first run may build the font cache)...", flush=True)
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

print("Importing scikit-learn...", flush=True)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Scientific libraries ready.")

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

### Reusable helpers

These helpers implement the pieces that every k-NN experiment needs: a pipeline that
scales (and optionally applies PCA) before classification, cross-validated tuning over
$k$, and a timing wrapper.

In [ ]:
def build_knn_pipeline(k, n_components=None, random_state=RANDOM_STATE):
    # Standardise, optionally reduce with PCA, then classify with k-NN.
    steps = [("scaler", StandardScaler())]
    if n_components is not None:
        steps.append(("pca", PCA(n_components=n_components, random_state=random_state)))
    steps.append(("knn", KNeighborsClassifier(n_neighbors=int(k), n_jobs=-1)))
    return Pipeline(steps)


def tune_k(X, y, ks, cv_splits=3, n_components=None, random_state=RANDOM_STATE):
    # Cross-validated accuracy for every candidate k.
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    rows = []
    for k in ks:
        model = build_knn_pipeline(k, n_components=n_components, random_state=random_state)
        scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
        rows.append(
            {"k": int(k), "mean": float(scores.mean()), "std": float(scores.std())}
        )
    return rows


def best_row(rows):
    return max(rows, key=lambda row: row["mean"])


def plot_k_curve(rows, title):
    ks = [row["k"] for row in rows]
    means = np.array([row["mean"] for row in rows])
    stds = np.array([row["std"] for row in rows])
    j = int(np.argmax(means))

    fig, ax = plt.subplots()
    ax.errorbar(ks, means, yerr=stds, marker="o", capsize=3)
    ax.axvline(ks[j], color="k", linestyle=":", label=f"best k = {ks[j]}")
    ax.set_xlabel("$k$")
    ax.set_ylabel("Cross-validated accuracy")
    ax.set_title(title)
    ax.legend()
    plt.show()


def fit_and_time(model, X_train, y_train, X_test, y_test):
    # Fit, predict, and report accuracy plus per-query prediction time.
    model.fit(X_train, y_train)
    model.predict(X_test[:5])  # warm-up
    start = perf_counter()
    y_pred = model.predict(X_test)
    elapsed = perf_counter() - start
    return {
        "model": model,
        "y_pred": y_pred,
        "acc": accuracy_score(y_test, y_pred),
        "seconds": elapsed,
        "ms_per_query": 1000.0 * elapsed / len(X_test),
    }

## 1. Load and inspect

Iris is built into scikit-learn, so no download is required. The four features are sepal
length, sepal width, petal length, and petal width, all in centimetres.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
class_names_iris = list(iris.target_names)

print("Source:   scikit-learn load_iris")
print(f"Samples:  {X_iris.shape[0]}")
print(f"Features: {X_iris.shape[1]} -> {iris.feature_names}")
print(f"Classes:  {len(class_names_iris)} -> {class_names_iris}")
print("Class counts:", np.bincount(y_iris))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for label, name in enumerate(class_names_iris):
    mask = y_iris == label
    axes[0].scatter(X_iris[mask, 0], X_iris[mask, 1], s=35, label=name)
    axes[1].scatter(X_iris[mask, 2], X_iris[mask, 3], s=35, label=name)
axes[0].set(xlabel="sepal length (cm)", ylabel="sepal width (cm)", title="Sepal features")
axes[1].set(xlabel="petal length (cm)", ylabel="petal width (cm)", title="Petal features")
axes[1].legend(title="species")
fig.tight_layout()
plt.show()

The petal measurements separate the classes far better than the sepal measurements.
$k$-NN uses all four features, but the plot already tells us that most of the signal
lives in petals 3 and 4.

## 2. Split first, then scale

The order matters. We split **before** computing the mean and standard deviation, so the
test set never influences preprocessing. This is the simplest form of leakage prevention.

Inside cross-validation the same rule is enforced automatically because the scaler lives
in a `Pipeline`: it is fitted on each training fold only.

In [ ]:
X_ir_train, X_ir_test, y_ir_train, y_ir_test = train_test_split(
    X_iris, y_iris, test_size=0.30, stratify=y_iris, random_state=RANDOM_STATE
)
print(f"Train: {X_ir_train.shape}  Test: {X_ir_test.shape}")

scaler_ir = StandardScaler().fit(X_ir_train)
X_ir_train_s = scaler_ir.transform(X_ir_train)
X_ir_test_s = scaler_ir.transform(X_ir_test)

print("Training means after scaling:", np.round(X_ir_train_s.mean(axis=0), 6))
print("Training stds  after scaling:", np.round(X_ir_train_s.std(axis=0), 6))

baseline = KNeighborsClassifier(n_neighbors=5).fit(X_ir_train_s, y_ir_train)
print(f"\n5-NN test accuracy (no tuning): {baseline.score(X_ir_test_s, y_ir_test):.3f}")

Standardisation matters here because the features have different ranges and units. A
distance computed on raw centimetres would be dominated by the largest-variance feature.

## 3. Choose $k$ with cross-validation

We try odd values from 1 to 31 with stratified 5-fold cross-validation on the training
set. Odd values avoid ties in a three-class vote. The test set stays untouched until the
next section.

In [ ]:
ks_iris = list(range(1, 32, 2))
rows_iris = tune_k(X_ir_train, y_ir_train, ks_iris, cv_splits=5)
best_iris = best_row(rows_iris)

plot_k_curve(rows_iris, "Iris: 5-fold cross-validation accuracy")

for row in rows_iris:
    print(f"k = {row['k']:2d}   CV accuracy = {row['mean']:.3f} +/- {row['std']:.3f}")

print(f"\nSelected k = {best_iris['k']} with CV accuracy {best_iris['mean']:.3f}")

The curve is flat over a wide range because Iris is easy. Small $k$ is slightly worse
(high variance), and very large $k$ begins to underfit. On a dataset this small, do not
read too much into one or two percentage points: the cross-validation standard deviation
is of the same order.

## 4. Final model and test set

Now we refit the selected $k$ on the full training set and evaluate once on the held-out
test set. This is the number to report.

In [ ]:
best_k_iris = best_iris["k"]
iris_model = build_knn_pipeline(best_k_iris)
iris_result = fit_and_time(
    iris_model, X_ir_train, y_ir_train, X_ir_test, y_ir_test
)
iris_acc = iris_result["acc"]
iris_ms = iris_result["ms_per_query"]

print(f"k = {best_k_iris}")
print(f"Test accuracy: {iris_acc:.3f}")
print(f"Prediction time: {iris_result['seconds'] * 1000:.2f} ms total, "
      f"{iris_ms:.3f} ms per query")
print()
print(classification_report(y_ir_test, iris_result["y_pred"],
                            target_names=class_names_iris, digits=3))

cm_iris = confusion_matrix(y_ir_test, iris_result["y_pred"])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_iris, display_labels=class_names_iris).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title(f"Iris: k-NN (k = {best_k_iris}) on the test set")
plt.show()

The confusion matrix is the useful diagnostic. On Iris the errors are almost always
versicolor versus virginica; setosa is never confused because it is far away in petal
space. Accuracy alone would hide that structure.

## 5. What the decision boundary looks like

The final model used all four features. To draw a boundary we refit on the two petal
features, which are the informative pair. This is a visual aid, not the final model, so
the accuracy printed below can differ slightly from the four-feature result.

In [ ]:
feature_idx = [2, 3]  # petal length and petal width
X2_train = X_ir_train[:, feature_idx]
X2_test = X_ir_test[:, feature_idx]

boundary_model = build_knn_pipeline(best_k_iris).fit(X2_train, y_ir_train)

x_min, x_max = X2_train[:, 0].min() - 0.5, X2_train[:, 0].max() + 0.5
y_min, y_max = X2_train[:, 1].min() - 0.5, X2_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400)
)
zz = boundary_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5, 2.5],
            colors=["#FFDDDD", "#DDFFDD", "#DDDDFF"], alpha=0.7)
ax.scatter(X2_train[:, 0], X2_train[:, 1], c=y_ir_train,
           cmap=ListedColormap(["#CC3311", "#0077BB", "#009988"]),
           edgecolor="k", s=35, label="train")
ax.scatter(X2_test[:, 0], X2_test[:, 1], c=y_ir_test, marker="X", s=90,
           cmap=ListedColormap(["#CC3311", "#0077BB", "#009988"]),
           edgecolor="white", linewidth=0.8, label="test")
ax.set(xlabel="petal length (cm)", ylabel="petal width (cm)",
       title=f"Iris decision regions (k = {best_k_iris}, two features)")
ax.legend()
plt.show()

print(f"Two-feature test accuracy: {boundary_model.score(X2_test, y_ir_test):.3f}")

The boundary is piecewise linear, made of small Voronoi-like cells. With $k>1$ the cells
smooth out. Iris is easy enough that even this two-feature view is nearly perfect.

**Takeaways**

- scaling is essential when features have different units;
- $k$ is not very sensitive here, but cross-validation still gives a defensible choice;
- a tiny dataset gives noisy estimates, so report standard deviations;
- the confusion matrix shows *which* species are confused, not just how often.

## What to try next

1. **Feature pairs.** Repeat the boundary plot with the sepal features. How much worse is
   the two-feature model?
2. **Distance weighting.** Add `weights="distance"` to the classifier and rerun. Does it
   change the selected $k$ or the test accuracy?
3. **Leave-one-out.** Replace the 5-fold CV with `LeaveOneOut()` on such a small dataset.
   How stable is the chosen $k$?
4. **Scaling ablation.** Fit $k$-NN on the raw (unscaled) features. Which class suffers
   most, and why?
5. **Repeated splits.** Run the whole workflow with 20 different random splits and report
   the mean and standard deviation of test accuracy.

## Further reading

- T. Cover and P. Hart, "Nearest Neighbor Pattern Classification", *IEEE Transactions on
  Information Theory*, 1967.
- R. A. Fisher, "The use of multiple measurements in taxonomic problems", *Annals of
  Eugenics*, 1936.
- scikit-learn user guide, *Nearest Neighbors*.